# Smart Waste Segregation - classifier training on Colab

Trains and evaluates the 6-class TrashNet classifiers on a Colab GPU, then saves the
results back to Google Drive.

**Before you run this**, make sure of two things (see `docs/COLAB.md`):

1. `src/train_classifier.py` and `src/evaluate.py` are **committed and pushed** to GitHub -
   this notebook gets them by cloning the repo.
2. The TrashNet images are on your Drive (a zip is much faster than a folder).

**Runtime -> Change runtime type -> GPU** before starting.

Order of cells: config -> mount Drive -> clone + install -> copy data -> run configs -> zip to Drive.

## 0. Configuration

Everything you might need to change lives in this one cell.

In [ ]:
# ---- your repo ---------------------------------------------------------------
REPO_URL  = "https://github.com/aarya5p/smart-waste-management.git"
BRANCH    = "main"
REPO_IS_PRIVATE = False      # True -> you will be prompted for a GitHub token

# ---- where your data lives on Drive -------------------------------------------
# Either a .zip of the class folders (FAST, recommended) or the folder itself (slow).
# The zip/folder must contain the 6 class folders: cardboard, glass, metal, paper, plastic, trash
DRIVE_TRASHNET   = "/content/drive/MyDrive/minor_project/dataset-resized.zip"

# Where the result zips get written back to
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/minor_project/outputs"

# ---- local (fast) paths on the Colab VM ---------------------------------------
WORK_DIR  = "/content"
DATA_ROOT = "/content/data/raw/trashnet"     # the 6 class folders end up here

# ---- training defaults (per-config values below can override these) -----------
EPOCHS      = 30
BATCH_SIZE  = 32
NUM_WORKERS = 2        # Colab VMs usually have 2 vCPUs

# Every run is scored on BOTH splits, into results/<run_name>/val/ and results/<run_name>/test/.
# val  = the ONLY split you are allowed to make decisions from
# test = reported in the final tables, never used to change anything
EVAL_SPLITS = {"val": "splits/val.csv", "test": "splits/test.csv"}

# ---- dependency install strategy ----------------------------------------------
# Colab already ships torch/torchvision built for its exact CUDA driver, plus numpy,
# pandas, matplotlib, scipy and scikit-learn. Reinstalling the pinned versions from
# requirements.txt is slow and can break the CUDA build, so by default we install
# ONLY the packages Colab is missing. Set STRICT_REQUIREMENTS = True to force the
# exact pins from requirements.txt instead (slow, may require a runtime restart).
STRICT_REQUIREMENTS = False
EXTRA_PACKAGES = ["timm", "ultralytics", "imagehash", "grad-cam", "fvcore"]

import os
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
REPO_DIR  = os.path.join(WORK_DIR, REPO_NAME)
print("repo dir :", REPO_DIR)
print("data root:", DATA_ROOT)

In [ ]:
import subprocess, sys, time
from pathlib import Path

def run(cmd, cwd=None, check=True):
    """Run a command and stream its output live (so you can watch training progress)."""
    print("$", " ".join(str(c) for c in cmd), flush=True)
    t0 = time.time()
    p = subprocess.Popen([str(c) for c in cmd], cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    dt = time.time() - t0
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode} after {dt:.0f}s")
    print(f"[done in {dt:.0f}s]\n", flush=True)
    return p.returncode

## 1. Mount Google Drive and check the GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.")

## 2. Clone the repo and install dependencies

Re-running this cell pulls the latest commit instead of cloning again, so you can push a
fix from your laptop and just re-run it.

In [ ]:
import shutil

clone_url = REPO_URL
if REPO_IS_PRIVATE:
    from getpass import getpass
    token = getpass("GitHub personal access token (not stored, not printed): ")
    clone_url = REPO_URL.replace("https://", f"https://{token}@")

if Path(REPO_DIR, ".git").exists():
    print("Repo already present - pulling latest.")
    run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)
else:
    run(["git", "clone", "--branch", BRANCH, clone_url, REPO_DIR], cwd=WORK_DIR)

del clone_url    # don't leave a tokenised URL lying around in the kernel

run(["git", "log", "-1", "--oneline"], cwd=REPO_DIR)

# Fail early and clearly if the training scripts were never pushed.
required = ["src/train_classifier.py", "src/evaluate.py",
            "splits/train.csv", "splits/val.csv", "splits/test.csv"]
missing = [f for f in required if not Path(REPO_DIR, f).exists()]
if missing:
    raise SystemExit(
        "These files are missing from the clone:\n  " + "\n  ".join(missing) +
        "\n\nCommit and push them from your laptop first:\n"
        "    git add src/ splits/ && git commit -m 'add training scripts' && git push"
    )
print("\nAll required files present.")

In [ ]:
if STRICT_REQUIREMENTS:
    # Exact pins from requirements.txt. Slow, and may force a runtime restart.
    run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=REPO_DIR)
else:
    # Only what Colab is missing; leave its CUDA-matched torch stack alone.
    run([sys.executable, "-m", "pip", "install", "-q", *EXTRA_PACKAGES], cwd=REPO_DIR)

# Record exactly what is installed, so the numbers in your report are traceable
# to a specific environment (Colab's versions differ from your laptop's).
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
Path(REPO_DIR, "pip_freeze_colab.txt").write_text(freeze)

import importlib
for mod in ["torch", "torchvision", "timm", "sklearn", "fvcore"]:
    try:
        m = importlib.import_module(mod)
        print(f"{mod:12s} {getattr(m, '__version__', 'n/a')}")
    except ImportError as e:
        print(f"{mod:12s} NOT INSTALLED ({e})")

## 3. Copy TrashNet from Drive to local disk

Training directly off `/content/drive` is slow: every image read goes through a network
filesystem, which starves the GPU. Copying the ~43 MB dataset to the VM's local disk once
takes a few seconds and makes every epoch faster.

In [ ]:
import zipfile

CLASSES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
src = Path(DRIVE_TRASHNET)
dst = Path(DATA_ROOT)

if not src.exists():
    raise SystemExit(f"Not found on Drive: {src}\nFix DRIVE_TRASHNET in the config cell.")

if dst.exists() and all((dst / c).is_dir() for c in CLASSES):
    print("Dataset already on local disk - skipping copy.")
else:
    dst.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    if src.suffix.lower() == ".zip":
        print(f"Unzipping {src} ...")
        with zipfile.ZipFile(src) as z:
            z.extractall("/content/_trashnet_tmp")
        # The zip may hold the class folders directly, or nest them one level down
        # (e.g. dataset-resized/cardboard/...). Find whichever level has the classes.
        root = None
        for cand in [Path("/content/_trashnet_tmp"), *Path("/content/_trashnet_tmp").iterdir()]:
            if cand.is_dir() and all((cand / c).is_dir() for c in CLASSES):
                root = cand
                break
        if root is None:
            raise SystemExit("Could not find the 6 class folders inside the zip.")
        for c in CLASSES:
            shutil.copytree(root / c, dst / c, dirs_exist_ok=True)
        shutil.rmtree("/content/_trashnet_tmp", ignore_errors=True)
    else:
        print(f"Copying folder {src} ... (slow over Drive)")
        for c in CLASSES:
            if not (src / c).is_dir():
                raise SystemExit(f"Missing class folder on Drive: {src/c}")
            shutil.copytree(src / c, dst / c, dirs_exist_ok=True)
    print(f"Copied in {time.time()-t0:.0f}s")

# Drop macOS junk that would otherwise be scanned as images
for junk in dst.rglob(".DS_Store"):
    junk.unlink()

total = 0
for c in CLASSES:
    n = len(list((dst / c).glob("*.jpg")))
    total += n
    print(f"  {c:10s} {n:4d}")
print(f"  {'TOTAL':10s} {total:4d}   (expected 2527)")

In [ ]:
# Sanity gate: every path in the committed splits must exist locally.
# Catches a wrong Drive path or a half-finished copy BEFORE you burn GPU time.
import pandas as pd

for split in ["train", "val", "test"]:
    df = pd.read_csv(Path(REPO_DIR, "splits", f"{split}.csv"))
    missing = [p for p in df["filepath"] if not (Path(DATA_ROOT) / p).exists()]
    print(f"{split:5s} {len(df):5d} rows | missing files: {len(missing)}")
    if missing:
        raise SystemExit(f"{len(missing)} files from {split}.csv are not on disk, "
                         f"e.g. {missing[:3]}")
print("\nAll split files present - safe to train.")

## 4. Define the runs

Each entry becomes one `train_classifier.py` run followed by one `evaluate.py` run per split
(val and test). `model`, `mode` and `seed` are required; `epochs`, `batch_size`, `lr` and
`img_size` are optional per-config overrides.

CLAUDE.md asks for seeds 42, 56 and 99 for every classifier config (rule 5) - the commented
block below is the full Phase 3 sweep.

In [ ]:
CONFIGS = [
    {"model": "mobilenet_v2", "mode": "frozen", "seed": 42},
]

# --- Phase 3 full sweep: all 5 classifiers, frozen, 3 seeds (uncomment when ready) ---
# CONFIGS = [
#     {"model": m, "mode": "frozen", "seed": s}
#     for m in ["mobilenet_v2", "mobilenet_v3_large", "efficientnet_b0", "resnet50", "convnext_tiny"]
#     for s in [42, 56, 99]
# ]

# --- fine-tuned subset (2-3 models, per CLAUDE.md) ---
# CONFIGS += [
#     {"model": m, "mode": "finetune", "seed": s}
#     for m in ["mobilenet_v2", "efficientnet_b0"]
#     for s in [42, 56, 99]
# ]

for c in CONFIGS:
    print(f"{c['model']}_{c['mode']}_seed{c['seed']}")
print(f"\n{len(CONFIGS)} run(s) queued")

### Run them

Training uses `splits/train.csv` + `splits/val.csv`; early stopping watches validation loss
only. Each trained model is then scored on **both** splits:

- `results/<run_name>/val/`  - validation metrics
- `results/<run_name>/test/` - test metrics

> **Decision rule (CLAUDE.md rule 2).** Every decision - hyper-parameters, which models to
> fine-tune, which model to deploy - is made from the **validation** columns only. The test
> columns exist to be reported in the final tables and must never cause you to change
> anything. If a test number tempts you to revisit a choice, that is exactly the situation
> this rule exists to prevent.

In [ ]:
import json

results_summary = []

for i, cfg in enumerate(CONFIGS, 1):
    run_name = f"{cfg['model']}_{cfg['mode']}_seed{cfg['seed']}"
    print("=" * 78)
    print(f"[{i}/{len(CONFIGS)}] {run_name}")
    print("=" * 78, flush=True)

    train_cmd = [sys.executable, "src/train_classifier.py",
                 "--data_root", DATA_ROOT,
                 "--model", cfg["model"], "--mode", cfg["mode"], "--seed", cfg["seed"],
                 "--epochs", cfg.get("epochs", EPOCHS),
                 "--batch_size", cfg.get("batch_size", BATCH_SIZE),
                 "--num_workers", NUM_WORKERS]
    for opt in ["lr", "img_size", "patience"]:
        if opt in cfg:
            train_cmd += [f"--{opt}", cfg[opt]]
    run(train_cmd, cwd=REPO_DIR)

    ckpt = Path(REPO_DIR, "runs", run_name, "best.pt")
    if not ckpt.exists():
        raise SystemExit(f"No checkpoint produced at {ckpt}")

    # Score on every split into its own folder: results/<run_name>/<split>/
    row = {"run": run_name}
    for split, csv_path in EVAL_SPLITS.items():
        out_dir = f"results/{run_name}/{split}"
        print(f"--- evaluating on {split} ({csv_path}) ---", flush=True)
        run([sys.executable, "src/evaluate.py",
             "--checkpoint", f"runs/{run_name}/best.pt",
             "--data_root", DATA_ROOT,
             "--csv", csv_path,
             "--out_dir", out_dir,
             "--num_workers", NUM_WORKERS], cwd=REPO_DIR)

        metrics_path = Path(REPO_DIR, out_dir, "metrics.json")
        if metrics_path.exists():
            m = json.loads(metrics_path.read_text())
            row[f"{split}_acc"] = round(m["accuracy"], 4)
            row[f"{split}_macro_f1"] = round(m["macro_f1"], 4)
    results_summary.append(row)

if results_summary:
    print("\nSummary of this session:")
    print(pd.DataFrame(results_summary).to_string(index=False))
    print("\nDecide from the val_* columns ONLY. The test_* columns are for the final "
          "report tables and must not change any choice you make.")

## 5. Zip `runs/` + `results/` and save to Drive

The Colab VM is wiped when the session ends, so anything you want to keep has to go to Drive.
The zip also contains `LOG.md` (with the rows the scripts appended, per rule 6) and
`pip_freeze_colab.txt` (exactly which library versions produced these numbers).

In [ ]:
import zipfile
from datetime import datetime

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = Path(WORK_DIR, f"waste_results_{stamp}.zip")

include_dirs  = ["runs", "results"]
include_files = ["LOG.md", "pip_freeze_colab.txt"]

n = 0
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for d in include_dirs:
        for p in Path(REPO_DIR, d).rglob("*"):
            if p.is_file():
                z.write(p, p.relative_to(REPO_DIR))
                n += 1
    for f in include_files:
        p = Path(REPO_DIR, f)
        if p.exists():
            z.write(p, p.relative_to(REPO_DIR))
            n += 1

print(f"{n} files -> {zip_path} ({zip_path.stat().st_size/1024**2:.1f} MB)")

out_dir = Path(DRIVE_OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
dest = out_dir / zip_path.name
shutil.copy2(zip_path, dest)
print("Saved to Drive:", dest)

## After the run

1. Download / open the zip from `DRIVE_OUTPUT_DIR`.
2. Copy the new rows out of its `LOG.md` into your repo's `LOG.md` and commit them -
   the Colab VM's copy disappears with the session (rule 6). There are two eval rows per
   run, one per split; the Settings column says which (`eval on splits/val.csv` / `test.csv`).
3. Commit the contents of `results/` to the repo. Each run has:
   - `results/<run_name>/val/`  - metrics.json, confusion_matrix.png, predictions.csv
   - `results/<run_name>/test/` - same three files
   `runs/` holds the checkpoints and stays untracked by design.

**Latency caveat (rule 8):** any latency in `metrics.json` from this notebook was measured on
a Colab GPU, and `evaluate.py` stamps it `counts_for_report: false`. Only Raspberry Pi
measurements belong in the report's latency tables.